# Empirical Dataset Audit & Research Feasibility Analysis
**Project**: AI-Assisted Technical Candidate Assessment & Cross-Source Consistency Verification System  
**University Final Year Research Project**  
**Date**: August 2026

This notebook provides the complete empirical data audit, quality analysis, statistical distributions, and comparative evaluation across all 6 project datasets:
1. **`01_github_technical_roles`**: Developer multi-label technical role classifications & package dependency profiles (1,662 developers, 1,531 features).
2. **`02_github_expertise`**: Zenodo empirical software engineering replication package on library usage, temporal intervals, and expertise survey ratings (575 records).
3. **`03_resume_corpus`**: Large-scale Indeed IT resume corpus with annotated skills and role taxonomies (29,783 resumes).
4. **`04_esco`**: European Commission standard taxonomy of occupations, skills, and requirement relations (13,939 skills, 3,039 occupations, 129,004 relations).
5. **`05_fever`**: Fact Extraction and VERification benchmark for NLI claim entailment (145,449 claims).
6. **`06_averitec`**: Real-world claim verification benchmark with multi-hop QA decomposition and textual justifications (3,568 claims).
7. **Comparative Ablation Experiment**: Empirical evaluation of Model A (Resume-Only), Model B (Resume + Raw GitHub), and Model C (Proposed VeritasAI Cross-Source System).

In [ ]:
import os
import json
import zipfile
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

print('All statistical and NLP libraries imported successfully.')

---
## 1. Dataset 01: GitHub Technical Roles & Dependency Manifests (`01_github_technical_roles`)
**Source**: MSR / ICSE Developer Technical Role Benchmark  
**Purpose**: Establishes developer role priors and maps 798 real-world package dependency usage indicators.

In [ ]:
df_roles = pd.read_csv('datasets/01_github_technical_roles/processed_ground_truth.csv', sep=';')
print(f'Dimensions: {df_roles.shape[0]} developers x {df_roles.shape[1]} features')
print(f'Missing Values: {df_roles.isnull().sum().sum()}')
print(f'Duplicate Rows: {df_roles.duplicated().sum()}')

targets = ['Backend', 'Frontend', 'Mobile', 'DevOps', 'DataScientist']
target_counts = df_roles[targets].sum()
print('\n--- Target Class Distribution ---')
for t, c in target_counts.items():
    print(f'  {t:15s}: {c:4d} positive ({c / len(df_roles) * 100:.1f}%)')

full_stack_count = len(df_roles[(df_roles['Backend'] == 1) & (df_roles['Frontend'] == 1)])
print(f'\nFull-Stack Developers (Both Backend & Frontend = 1): {full_stack_count} ({full_stack_count / len(df_roles) * 100:.1f}%)')

bio_cols = [c for c in df_roles.columns if '(Bio)' in c]
dep_cols = [c for c in df_roles.columns if '(dep.)' in c]
desc_cols = [c for c in df_roles.columns if '(desc.)' in c]
print(f'\nFeature Breakdown: {len(bio_cols)} Bio TF-IDF | {len(dep_cols)} Package Dependencies | {len(desc_cols)} Repo Topics/Language Features')
print(f'Sample Dependencies Mapped: {dep_cols[:8]}')

---
## 2. Dataset 02: Empirical Developer Expertise & Temporal Metrics (`02_github_expertise`)
**Source**: Zenodo Empirical Software Engineering Replication Package  
**Purpose**: Eliminates arbitrary skill scoring by grounding temporal intervals (`daysSinceFirstImport`, `daysSinceLastImport`) and commit volumes in human survey ratings (1-5).

In [ ]:
expertise_files = ['react.csv', 'mongodb.csv', 'socketio.csv']
expertise_dfs = []
print('--- Expertise Dataset Summaries ---')
for ef in expertise_files:
    path = os.path.join('datasets/02_github_expertise', ef)
    df = pd.read_csv(path, sep=';', encoding='latin1')
    df['technology'] = ef.replace('.csv', '')
    expertise_dfs.append(df)
    print(f'  {ef:15s}: {len(df):3d} developers | Mean Survey Rating: {df["survey"].mean():.2f} / 5.0 | Commits Range: [{df["commits"].min()} - {df["commits"].max()}]')

combined_expertise = pd.concat(expertise_dfs, ignore_index=True)
print(f'\nTotal Developer-Technology Records: {len(combined_expertise)}')
print('\nCorrelation with Human Survey Expertise Rating:')
corrs = combined_expertise[['commits', 'commitsImportLibrary', 'imports', 'daysSinceFirstImport', 'daysSinceLastImport', 'projectsImport', 'survey']].corr()['survey']
print(corrs.to_frame(name='Correlation'))

---
## 3. Dataset 03: Indeed IT Resume Corpus (`03_resume_corpus`)
**Source**: Indeed Large-Scale IT Resume Corpus  
**Purpose**: Real-world candidate resumes with annotated skills and standardized occupational taxonomies.

In [ ]:
with zipfile.ZipFile('datasets/03_resume_corpus/resumes_corpus.zip', 'r') as z:
    lab_files = [f for f in z.namelist() if f.endswith('.lab')]
    class_counter = Counter()
    for lf in lab_files:
        cls = z.read(lf).decode('utf-8', errors='ignore').strip()
        class_counter[cls] += 1
    print(f'Total Resumes in Corpus: {len(lab_files):,}')
    print('\nTop 10 Resume Class Distribution:')
    for cls, count in class_counter.most_common(10):
        clean_cls = cls.replace('\n', ' / ')
        print(f'  {clean_cls:45s}: {count:5d} resumes')

with zipfile.ZipFile('datasets/03_resume_corpus/resume_samples.zip', 'r') as z:
    with z.open('resume_samples.txt') as f:
        first_line = f.readline().decode('utf-8', errors='ignore')
        parts = first_line.strip().split(':::')
        print(f'\nSample Structured Resume Record:')
        print(f'  Meta ID:   {parts[0]}')
        print(f'  Skills:    {parts[1][:120]}...')
        print(f'  Text Span: {parts[2][:150]}...')

---
## 4. Dataset 04: ESCO v1.1 Standard European Taxonomy (`04_esco`)
**Source**: European Commission ESCO Standard Competency Taxonomy  
**Purpose**: Ground-truth knowledge graph for occupations, essential vs. optional skill relations, and canonical alias expansion.

In [ ]:
df_occ = pd.read_csv('datasets/04_esco/occupations_en.csv')
df_sk = pd.read_csv('datasets/04_esco/skills_en.csv')
df_rel = pd.read_csv('datasets/04_esco/occupationSkillRelations_en.csv')

print(f'Total ESCO Occupations: {len(df_occ):,}')
print(f'Total ESCO Skills:      {len(df_sk):,}')
print(f'Total Skill Relations:  {len(df_rel):,} ({df_rel["relationType"].value_counts().to_dict()})')

swe_occ = df_occ[df_occ['preferredLabel'] == 'software developer'].iloc[0]
swe_uri = swe_occ['conceptUri']
swe_rels = df_rel[df_rel['occupationUri'] == swe_uri]
sk_map = df_sk.set_index('conceptUri')['preferredLabel'].to_dict()

essential_swe = [sk_map.get(u, 'Unknown') for u in swe_rels[swe_rels['relationType'] == 'essential']['skillUri']]
optional_swe = [sk_map.get(u, 'Unknown') for u in swe_rels[swe_rels['relationType'] == 'optional']['skillUri']]

print(f'\nSample "Software Developer" ESCO Profile:')
print(f'  Essential Skills ({len(essential_swe)}): {essential_swe[:6]}')
print(f'  Optional Skills  ({len(optional_swe)}): {optional_swe[:6]}')

---
## 5. Datasets 05 & 06: Fact-Checking & Claim Verification Baselines (`05_fever` & `06_averitec`)
**Purpose**: Methodological blueprint for 4-way NLI epistemic formulation and multi-hop textual justifications.

In [ ]:
# FEVER Label Breakdown
fever_labels = {}
fever_total = 0
with open('datasets/05_fever/train.jsonl', 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        fever_total += 1
        lbl = json.loads(line).get('label')
        fever_labels[lbl] = fever_labels.get(lbl, 0) + 1

print(f'FEVER Benchmark ({fever_total:,} claims):')
for k, v in fever_labels.items():
    print(f'  {k:18s}: {v:6d} ({v/fever_total*100:.1f}%)')

# AVeriTeC Label Breakdown
with open('datasets/06_averitec/train.json', 'r', encoding='utf-8', errors='ignore') as f:
    av_train = json.load(f)
av_labels = {}
for item in av_train:
    lbl = item.get('label')
    av_labels[lbl] = av_labels.get(lbl, 0) + 1

print(f'\nAVeriTeC Real-World Benchmark ({len(av_train):,} claims):')
for k, v in av_labels.items():
    print(f'  {k:36s}: {v:5d} ({v/len(av_train)*100:.1f}%)')

---
## 6. Scientific Ablation Benchmark Study: Model A vs Model B vs Model C
**Experiment**: Evaluated on $N = 300$ balanced candidate instances ($1,275$ individual claim-evidence evaluations).

In [ ]:
with open('results/ablation_study_results.json', 'r') as f:
    ablation_data = json.load(f)

summary_rows = []
for model_name, res in ablation_data.items():
    cls = res['classification']
    cal = res['calibration']
    summary_rows.append({
        'Model Configuration': model_name,
        'Accuracy': f"{cls['accuracy'] * 100:.2f}%",
        'Macro-F1': f"{cls['macroF1']:.4f}",
        'Calibration Error (ECE)': f"{cal['ece']:.4f}",
        'Brier Score': f"{cal['brierScore']:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
print('=== COMPARATIVE ABLATION STUDY RESULTS ===')
print(summary_df.to_string(index=False))

print('\n=== PER-CLASS F1 BREAKDOWN (MODEL C) ===')
model_c_res = ablation_data['Model C (Full Proposed Cross-Source)']['classification']
for c_name, scores in model_c_res['perClass'].items():
    print(f'  {c_name:24s}: F1 = {scores["f1"]:.4f} | Precision = {scores["precision"]:.4f} | Recall = {scores["recall"]:.4f} (Support = {scores["support"]})')